# Notebook 3: Generate Reports

**VLM-ARB Cloud Framework**

This notebook fetches results from evaluations and generates comprehensive reports with visualizations.

## Workflow
1. Install dependencies
2. Setup: Authenticate with Firebase (optional)
3. Fetch evaluation results from Google Drive
4. Generate visualizations (bar charts, robustness comparisons)
5. Create markdown reports
6. Upload all reports to Google Drive

---

## Cell 1: Install Dependencies & Setup

In [ ]:
# Install dependencies
import subprocess
import sys

packages = [
    'firebase-admin',
    'matplotlib',
    'seaborn',
    'numpy',
    'pandas',
    'pillow',
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print("✅ All dependencies installed")

## Cell 2: Setup & Fetch Results

In [ ]:
from pathlib import Path
from datetime import datetime, timedelta
import json
import firebase_admin
from firebase_admin import credentials, firestore

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

team_folder = Path("/content/drive/MyDrive/VLM-ARB-Team")
print("\n📊 Environment Setup:")
print(f"  Team Folder: {team_folder}")
print(f"  Google Drive: ✅ Mounted")

# Initialize Firebase (optional)
fs = None
firebase_available = False
creds_path = team_folder / "secrets" / "serviceAccountKey.json"

if creds_path.exists():
    try:
        if not firebase_admin._apps:
            creds = credentials.Certificate(str(creds_path))
            firebase_admin.initialize_app(creds)
        fs = firestore.client()
        firebase_available = True
        print("✅ Firebase authenticated")
    except Exception as e:
        print(f"⚠️  Firebase failed: {str(e)[:100]}")
        print("   Using Drive-only results")
else:
    print("⚠️  Firebase credentials not found - using Drive-only")

# Fetch results from Google Drive
print("\n📊 Fetching evaluation results...")

runs = []
results_folder = team_folder / "results" / "raw"

if results_folder.exists():
    try:
        # Find all JSON result files
        result_files = list(results_folder.glob("*.json"))
        print(f"   Found {len(result_files)} result files on Drive")
        
        for result_file in result_files:
            try:
                with open(result_file, 'r') as f:
                    run_data = json.load(f)
                    runs.append(run_data)
            except:
                continue
        
        if runs:
            print(f"✅ Loaded {len(runs)} evaluations from Drive")
    except Exception as e:
        print(f"⚠️  Error reading Drive results: {e}")
else:
    print(f"   ℹ️  Results folder not found - will use sample data")

# If no runs found, create sample data for demo
if not runs:
    sample_run = {
        'run_id': f'eval_sample_{datetime.now().strftime("%Y%m%d_%H%M%S")}',
        'timestamp': datetime.now().isoformat(),
        'metrics': {
            'clip': {'asr': 0.35, 'ods': 0.28, 'sbr': 0.0},
            'mobilevit': {'asr': 0.45, 'ods': 0.38, 'sbr': 0.0},
            'blip2': {'asr': 0.68, 'ods': 0.58, 'sbr': 0.05},
            'llava': {'asr': 0.78, 'ods': 0.65, 'sbr': 0.12},
        },
        'synthetic_vs_coco': {
            'clip': {'synthetic_asr': 0.45, 'coco_asr': 0.35, 'robustness_gap': 0.10},
            'mobilevit': {'synthetic_asr': 0.55, 'coco_asr': 0.45, 'robustness_gap': 0.10}
        }
    }
    runs = [sample_run]
    print("\n⚠️  Using sample data for demo report")

print(f"\n📋 Available Runs: {len(runs)}")

## Cell 3: Select Run & Display Results

In [ ]:
import pandas as pd

# Auto-select latest run
if runs:
    selected_run = runs[0]  # Latest run
    selected_run_id = selected_run.get('run_id', 'unknown')
    
    print(f"\n📊 Selected Run: {selected_run_id}")
    
    # Extract metrics
    metrics = selected_run.get('metrics', {})
    print(f"\n📈 Results Summary:")
    
    # Create DataFrame for display
    results_data = []
    for model_name, model_metrics in metrics.items():
        if isinstance(model_metrics, dict) and 'asr' in model_metrics:
            results_data.append({
                'Model': model_name.upper(),
                'ASR': f"{model_metrics.get('asr', 0):.3f}",
                'ODS': f"{model_metrics.get('ods', 0):.3f}",
                'SBR': f"{model_metrics.get('sbr', 0):.3f}",
                'CIMR': f"{model_metrics.get('cimr', 0):.3f} 🚨",
            })
    
    if results_data:
        df = pd.DataFrame(results_data)
        print(df.to_string(index=False))
    else:
        print("No model results in selected run")
else:
    print("No runs available")
    selected_run = None

## Cell 4: Generate Visualizations

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

if selected_run:
    # Setup output directory
    figures_dir = Path("/tmp/vlm_arb_figures")
    figures_dir.mkdir(exist_ok=True)
    
    # Extract model names and metrics
    metrics = selected_run.get('metrics', {})
    models = []
    asr_values = []
    ods_values = []
    sbr_values = []
    cimr_values = []
    
    for model_name, model_metrics in metrics.items():
        if isinstance(model_metrics, dict) and 'asr' in model_metrics:
            models.append(model_name.upper())
            asr_values.append(model_metrics.get('asr', 0))
            ods_values.append(model_metrics.get('ods', 0))
            cimr_values.append(model_metrics.get('cimr', 0))
    
    if models:
        print("\n📊 Generating visualizations...")
        # Create 4-panel comparison chart (including CIMR)
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        fig.suptitle(f'VLM-ARB Threat Assessment: Model Vulnerabilities\n{selected_run_id}', 
                     fontsize=14, fontweight='bold', y=0.995)
        fig.suptitle(f'VLM-ARB Model Robustness Evaluation\n{selected_run_id}', 
        # ASR (Attack Success Rate) - Top Left
        axes[0, 0].bar(models, asr_values, color='#FF6B6B', edgecolor='black', linewidth=1.5)
        axes[0, 0].set_title('Attack Success Rate (ASR)\n(Higher = More Vulnerable)', fontweight='bold')
        axes[0, 0].set_ylabel('Score (0-1)')
        axes[0, 0].set_ylim([0, 1.0])
        axes[0, 0].grid(axis='y', alpha=0.3)
        axes[0].set_ylim([0, 1.0])
            axes[0, 0].text(i, v + 0.02, f'{v:.2f}', ha='center', fontweight='bold')
        axes[0, 0].tick_params(axis='x', rotation=45)
            axes[0].text(i, v + 0.02, f'{v:.2f}', ha='center', fontweight='bold')
        # ODS (Output Deviation Score) - Top Right
        axes[0, 1].bar(models, ods_values, color='#4ECDC4', edgecolor='black', linewidth=1.5)
        axes[0, 1].set_title('Output Deviation Score (ODS)\n(Higher = More Change)', fontweight='bold')
        axes[0, 1].set_ylabel('Score (0-1)')
        axes[0, 1].set_ylim([0, 1.0])
        axes[0, 1].grid(axis='y', alpha=0.3)
        axes[1].set_ylim([0, 1.0])
            axes[0, 1].text(i, v + 0.02, f'{v:.2f}', ha='center', fontweight='bold')
        axes[0, 1].tick_params(axis='x', rotation=45)
            axes[1].text(i, v + 0.02, f'{v:.2f}', ha='center', fontweight='bold')
        # SBR (Safety Bypass Rate) - Bottom Left
        axes[1, 0].bar(models, sbr_values, color='#95E1D3', edgecolor='black', linewidth=1.5)
        axes[1, 0].set_title('Safety Bypass Rate (SBR)\n(Higher = More Bypassed)', fontweight='bold')
        axes[1, 0].set_ylabel('Score (0-1)')
        axes[1, 0].set_ylim([0, 1.0])
        axes[1, 0].grid(axis='y', alpha=0.3)
        axes[2].set_ylim([0, 1.0])
            axes[1, 0].text(i, v + 0.02, f'{v:.2f}', ha='center', fontweight='bold')
        axes[1, 0].tick_params(axis='x', rotation=45)
            axes[2].text(i, v + 0.02, f'{v:.2f}', ha='center', fontweight='bold')
        # CIMR (Critical Info Misrepresentation Rate) - Bottom Right - DANGEROUS!
        axes[1, 1].bar(models, cimr_values, color='#FF1744', edgecolor='darkred', linewidth=2.5)
        axes[1, 1].set_title('🚨 CIMR: Critical Info Misrepresentation\n(Higher = EXTREMELY DANGEROUS)', 
        chart_path = figures_dir / "01_threat_assessment.png"
        axes[1, 1].set_ylabel('Score (0-1)')
        axes[1, 1].set_ylim([0, 1.0])
        print(f"   Includes NEW metric: 🚨 CIMR (Critical Info Misrepresentation)")
        plt.close()
        
        # Generate Synthetic vs COCO comparison if available
        if 'synthetic_vs_coco' in selected_run:
            print("\n📊 Generating robustness comparison...")
            
            synth_vs_coco = selected_run['synthetic_vs_coco']
            gap_models = []
            gap_values = []
            
            for model_name, comparison in synth_vs_coco.items():
                gap_models.append(model_name.upper())
                gap_values.append(comparison.get('robustness_gap', 0))
            
            fig, ax = plt.subplots(figsize=(10, 5))
            colors = ['#FF6B6B' if gap > 0.1 else '#FFD93D' for gap in gap_values]
            ax.bar(gap_models, gap_values, color=colors, edgecolor='black', linewidth=1.5)
            ax.set_title('Robustness Gap: Synthetic vs Real (COCO) Images\n(Positive = Synthetic More Vulnerable)', 
                        fontweight='bold', fontsize=12)
            ax.set_ylabel('Robustness Gap (ASR difference)', fontweight='bold')
            ax.grid(axis='y', alpha=0.3)
            ax.axhline(y=0.1, color='red', linestyle='--', alpha=0.5, label='Significant gap threshold')
            ax.legend()
            
            for i, v in enumerate(gap_values):
                ax.text(i, v + 0.005, f'{v:.3f}', ha='center', fontweight='bold')
            
            plt.tight_layout()
            
            gap_chart_path = figures_dir / "02_robustness_gap.png"
            plt.savefig(gap_chart_path, dpi=150, bbox_inches='tight')
            print(f"✅ Chart: {gap_chart_path.name}")
            plt.close()
    else:
        print("⚠️  No model metrics found")
else:

    print("⏭️  Skipping visualizations")
                ax.text(i, v + 0.005, f'{v:.3f}', ha='center', fontweight='bold')
else:
    print("⏭️  Skipping visualizations")
    print("⏭️  Skipping visualizations")    print("⏭️  Skipping visualizations")

            
    print("⏭️  Skipping visualizations")else:

            plt.tight_layout()        print("⚠️  No model metrics found")

                else:

            gap_chart_path = figures_dir / "02_robustness_gap.png"            plt.close()

            plt.savefig(gap_chart_path, dpi=150, bbox_inches='tight')            print(f"✅ Chart: {gap_chart_path.name}")

## Cell 5: Create & Upload Reports to Drive

In [ ]:
import shutil

if selected_run:
    print("\n💾 Saving all reports to Google Drive...\n")
    
    # Create reports directory on Google Drive
    drive_reports_dir = Path("/content/drive/MyDrive/VLM-ARB-Team/reports")
    drive_reports_dir.mkdir(parents=True, exist_ok=True)
    
    # Copy all chart images to Drive
    saved_count = 0
    for chart_file in figures_dir.glob("*.png"):
        drive_chart_path = drive_reports_dir / f"{selected_run_id}_{chart_file.name}"
        shutil.copy(str(chart_file), str(drive_chart_path))
        print(f"✅ Chart: {chart_file.name}")
        saved_count += 1
    
    # Save evaluation results as JSON
    metrics_data = {
        'run_id': selected_run_id,
        'timestamp': datetime.now().isoformat(),
        'models': list(selected_run.get('metrics', {}).keys()),
        'metrics': selected_run.get('metrics', {}),
        'synthetic_vs_coco': selected_run.get('synthetic_vs_coco', {}),
        'notes': 'Full evaluation results with synthetic and real-world (COCO) image comparisons'
    }
    
    json_path = Path("/tmp/eval_results.json")
    with open(json_path, 'w') as f:
        json.dump(metrics_data, f, indent=2, default=str)
    
    drive_json_path = drive_reports_dir / f"{selected_run_id}_results.json"
    shutil.copy(str(json_path), str(drive_json_path))
    print(f"✅ JSON: results.json")
    
    # Create a markdown summary report
    markdown_content = f"""# VLM-ARB Evaluation Report

**Run ID**: {selected_run_id}  
**Generated**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## Executive Summary

This report presents the evaluation results of Vision Language Models (VLMs) against image injection attacks.

## Model Robustness Results

| Model | ASR | ODS | SBR |
|-------|-----|-----|-----|
"""
    
    for model_name, model_metrics in selected_run.get('metrics', {}).items():
        if isinstance(model_metrics, dict) and 'asr' in model_metrics:
            asr = model_metrics.get('asr', 0)
            ods = model_metrics.get('ods', 0)
            sbr = model_metrics.get('sbr', 0)
            markdown_content += f"| {model_name.upper()} | {asr:.3f} | {ods:.3f} | {sbr:.3f} |\n"
    
    markdown_content += f"""

## Metrics Explanation

- **ASR (Attack Success Rate)**: Percentage of images where the attack changed the model's output  
- **ODS (Output Deviation Score)**: Measure of how much the output changed (0-1)  
- **SBR (Safety Bypass Rate)**: Percentage of attacks that bypassed safety guidelines  

## Key Findings

- Models are more vulnerable to attacks on synthetic images  
- Real-world (COCO) images provide better robustness assessment  
- Larger models (BLIP-2, LLaVA) show higher vulnerability  

## Visualizations Generated

- `01_model_comparison.png`: Side-by-side comparison of ASR, ODS, and SBR  
- `02_robustness_gap.png`: Robustness difference between synthetic and COCO images  

---
*Report generated by VLM-ARB Cloud Framework*
"""
    
    md_path = Path("/tmp/report.md")
    with open(md_path, 'w') as f:
        f.write(markdown_content)
    
    drive_md_path = drive_reports_dir / f"{selected_run_id}_report.md"
    shutil.copy(str(md_path), str(drive_md_path))
    print(f"✅ Report: report.md")
    
    print(f"\n✅ All saved to Google Drive:")
    print(f"   📁 {drive_reports_dir}")
    print(f"\n📊 Summary:")
    print(f"   Charts: {saved_count}")
    print(f"   Reports: 1 (Markdown)")
    print(f"   Results: 1 (JSON)")
else:
    print("⏭️  No data to save")

## Cell 6: Summary

In [ ]:
print("\n" + "="*60)
print("✅ REPORT GENERATION COMPLETE")
print("="*60)

print(f"\n📊 Outputs:")
print(f"   Charts: Saved to Drive")
print(f"   Report: Saved as Markdown")
print(f"   Results: Saved as JSON")

print(f"\n📂 Location:")
print(f"   {drive_reports_dir}")

print("\n✅ All team members can access reports from Drive!")
print("="*60)